# Etapa 6: Modelado e Inferencia Estadística Dual (Paramétrica vs No Paramétrica)

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA & EVA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 6: Modeling / Evaluation) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, SWEBOK, DAMA-BOK, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Canal Paramétrico (SIPSA & EVA)**: Prueba de homogeneidad de varianzas (Levene), ANOVA de Welch para precios y rendimientos agrícolas departamentales.
2. **Canal No Paramétrico (Distribution-Free)**: Prueba de Kruskal-Wallis, contrastes post-hoc y remuestreo Bootstrap BCa (95%, $B=2,000$).
3. **Inferencia de Rendimiento Agronómico (EVA)**: Contrastar diferencias estadísticas de productividad agrícola (t/ha) entre las principales regiones productoras.


In [1]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
[OK] Ambiente verificado en Python: 3.13.5


In [2]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import FEATURES_DIR, CURATED_DIR
from src.application.model_service import ModelService

print("Servicio de Modelado Estadístico Dual cargado exitosamente.")

Servicio de Modelado Estadístico Dual cargado exitosamente.


In [3]:
# ==============================================================================
# CARGA DE DATOS DE LA CAPA FEATURES
# ==============================================================================
parquet_path = FEATURES_DIR / "dataset_sipsa_features.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con múltiples años para contrastes ANOVA y Kruskal-Wallis
    rows = []
    for año in [2021, 2022, 2023, 2024]:
        media_año = 1800 if año == 2021 else (2600 if año == 2022 else (2300 if año == 2023 else 2800))
        precios = np.random.gamma(shape=8.0, scale=media_año / 8.0, size=40)
        for p in precios:
            rows.append({
                "año": año,
                "precio_prom_kg": float(p),
                "volumen_ingreso_ton": float(np.random.uniform(50, 200)),
                "variedad_papa": "PAPA PASTUSA"
            })
    df = pd.DataFrame(rows)

print(f"Registros disponibles para contrastes de hipótesis: {len(df):,}")
display(df.head(4))

Registros disponibles para contrastes de hipótesis: 86,054


,fecha_mes,año,mes,divipola_depto,nombre_depto,divipola_mpio,nombre_mpio,mercado_mayorista,alimento,variedad_papa,...,precio_max_kg,num_transacciones,variedad_papa_original,poblacion_nacional_total,poblacion_nacional_cabecera,consumo_mayorista_per_capita_kg,indice_estacional_precio_iep,indice_estacional_oferta_ieo,es_outlier_iqr,es_outlier_mad
0,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"CALI, CAVASA",PAPA,PAPA NEVADA,...,2406.73,1,PAPA NEVADA,49266526,37219289,0.000215,93.493029,7.731589,False,False
1,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MEDELLIN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA COLOMBIA,...,3590.18,12,PAPA CRIOLLA,49266526,37219289,0.000117,139.459701,4.204052,False,False
2,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MONTERIA, MERCADO DEL SUR",PAPA,PAPA OTRAS VARIEDADES,...,2804.16,37,PAPA CAPIRA,49266526,37219289,0.002836,108.874934,102.008657,False,False
3,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLIN,"MONTERIA, MERCADO DEL SUR",PAPA,PAPA CRIOLLA COLOMBIA,...,3570.77,12,PAPA CRIOLLA,49266526,37219289,0.018539,137.609873,666.849580,False,False


In [4]:
# ==============================================================================
# EJECUCIÓN DE LA BATERÍA ESTADÍSTICA DUAL
# ==============================================================================
mod_service = ModelService(df, output_dir=CURATED_DIR)
df_tabla_dual, resumen = mod_service.compilar_y_guardar_resultados(col_grupo="año", col_valor="precio_prom_kg")

print("=== CONTRASTES PARAMÉTRICOS (ANOVA & LEVENE) ===")
print(f"Levene W: {resumen['canal_parametrico']['prueba_levene']['estadistico_W']:.3f} | p-valor: {resumen['canal_parametrico']['prueba_levene']['p_valor']:.4e}")
print(f"ANOVA F: {resumen['canal_parametrico']['anova_clasico']['estadistico_F']:.3f} | p-valor: {resumen['canal_parametrico']['anova_clasico']['p_valor']:.4e}")
print(f"Welch F: {resumen['canal_parametrico']['anova_welch']['estadistico_F_welch']:.3f} | p-valor: {resumen['canal_parametrico']['anova_welch']['p_valor']:.4e}")

print("\n=== CONTRASTES NO PARAMÉTRICOS (KRUSKAL-WALLIS & FLIGNER) ===")
print(f"Fligner Chi2: {resumen['canal_no_parametrico']['prueba_fligner_killeen']['estadistico_chi2']:.3f} | p-valor: {resumen['canal_no_parametrico']['prueba_fligner_killeen']['p_valor']:.4e}")
print(f"Kruskal H: {resumen['canal_no_parametrico']['kruskal_wallis']['estadistico_H']:.3f} | p-valor: {resumen['canal_no_parametrico']['kruskal_wallis']['p_valor']:.4e}")

print("\n=== TABLA COMPARATIVA: ESTIMADORES DUALES E INTERVALOS AL 95% ===")
display(df_tabla_dual)

=== CONTRASTES PARAMÉTRICOS (ANOVA & LEVENE) ===
Levene W: 902.209 | p-valor: 0.0000e+00
ANOVA F: 18250.191 | p-valor: 0.0000e+00
Welch F: 18995.604 | p-valor: 0.0000e+00

=== CONTRASTES NO PARAMÉTRICOS (KRUSKAL-WALLIS & FLIGNER) ===
Fligner Chi2: 5362.882 | p-valor: 0.0000e+00
Kruskal H: 51759.866 | p-valor: 0.0000e+00

=== TABLA COMPARATIVA: ESTIMADORES DUALES E INTERVALOS AL 95% ===


,grupo,n,media_parametrica,ic_t_student_inf_95,ic_t_student_sup_95,mediana_no_parametrica,ic_bootstrap_bca_inf_95,ic_bootstrap_bca_sup_95
0,2019,12313,2340.217263,2331.484162,2348.950364,2140.040,2331.513809,2348.524773
1,2020,12509,1926.850588,1919.668085,1934.033092,1764.960,1920.025338,1934.395800
2,2021,11771,2958.082843,2946.585229,2969.580456,2691.410,2946.508045,2969.086527
3,2022,11466,4322.306370,4305.154230,4339.458510,3919.960,4305.728851,4340.848281
4,2023,12636,3817.477522,3803.201282,3831.753762,3486.285,3803.659577,3831.801043
5,2024,12499,3031.287378,3019.956585,3042.618172,2756.890,3020.595291,3042.980801
6,2025,12860,2807.085149,2796.661919,2817.508378,2545.080,2796.670978,2817.702521


In [6]:
# ==============================================================================
# CONTRASTES ESTADÍSTICOS SOBRE RENDIMIENTOS AGRONÓMICOS (EVA: TON/HA)
# ==============================================================================
import importlib
import pandas as pd
from IPython.display import display
from src.infrastructure.config import CLEANED_DIR, CURATED_DIR

# Recargar dinámicamente el módulo para asegurar métodos agregados en caliente
import src.application.model_service
importlib.reload(src.application.model_service)
from src.application.model_service import ModelService

eva_path = CLEANED_DIR / "dataset_eva_agricola_nacional.parquet"
if eva_path.exists():
    df_eva = pd.read_parquet(eva_path)
    
    # Filtrar principales departamentos paperos
    top_deptos = ["CUNDINAMARCA", "BOYACÁ", "NARIÑO", "ANTIOQUIA"]
    df_eva_sub = df_eva[df_eva["departamento"].isin(top_deptos)].copy()
    
    # Ejecutar inferencia dual con ModelService sobre rendimiento_ton_ha
    mod_service_eva = ModelService(df_eva_sub, output_dir=CURATED_DIR)
    tabla_dual_eva, res_eva = mod_service_eva.compilar_y_guardar_resultados(
        col_grupo="departamento", 
        col_valor="rendimiento_ton_ha"
    )
    
    print("=== CONTRASTES ESTADÍSTICOS DE RENDIMIENTO AGRÍCOLA (EVA TON/HA) ===")
    print(f"Prueba de Homocedasticidad de Levene: W = {res_eva['canal_parametrico']['prueba_levene']['estadistico_W']:.3f} | p-valor = {res_eva['canal_parametrico']['prueba_levene']['p_valor']:.5e}")
    print(f"Prueba ANOVA de Welch: F-stat = {res_eva['canal_parametrico']['anova_welch']['estadistico_F_welch']:.4f} | p-valor = {res_eva['canal_parametrico']['anova_welch']['p_valor']:.5e}")
    print(f"Prueba Kruskal-Wallis (No paramétrica): H-stat = {res_eva['canal_no_parametrico']['kruskal_wallis']['estadistico_H']:.4f} | p-valor = {res_eva['canal_no_parametrico']['kruskal_wallis']['p_valor']:.5e}")
    
    print("\n=== TABLA COMPARATIVA DE ESTIMADORES DE RENDIMIENTO (t/ha) ===")
    display(tabla_dual_eva)



=== CONTRASTES ESTADÍSTICOS DE RENDIMIENTO AGRÍCOLA (EVA TON/HA) ===
Prueba de Homocedasticidad de Levene: W = 24.430 | p-valor = 1.13272e-15
Prueba ANOVA de Welch: F-stat = 117.7673 | p-valor = 0.00000e+00
Prueba Kruskal-Wallis (No paramétrica): H-stat = 383.1819 | p-valor = 9.72463e-83

=== TABLA COMPARATIVA DE ESTIMADORES DE RENDIMIENTO (t/ha) ===


,grupo,n,media_parametrica,ic_t_student_inf_95,ic_t_student_sup_95,mediana_no_parametrica,ic_bootstrap_bca_inf_95,ic_bootstrap_bca_sup_95
0,ANTIOQUIA,581,15.233873,14.534191,15.933554,14.0,14.540496,15.947475
1,BOYACÁ,1659,16.283400,15.965609,16.601190,16.0,15.976772,16.626822
2,CUNDINAMARCA,1347,19.671321,19.342831,19.999812,20.0,19.341622,19.992563
3,NARIÑO,792,15.231048,14.779584,15.682512,15.0,14.801087,15.697158


### Conclusiones de la Etapa 6:
- **Precios Mayoristas (SIPSA)**: La prueba de Levene rechaza homocedasticidad ($p < 0.05$), confirmando la validez del **ANOVA de Welch** y la prueba de **Kruskal-Wallis** para contrastar variaciones temporales y geográficas.
- **Rendimientos Agrícolas (EVA)**: Existen diferencias estadísticamente significativas ($p < 0.001$) entre los rendimientos agronómicos de Cundinamarca, Boyacá, Nariño y Antioquia, reflejando disparidades agroecológicas y de tecnificación.
- Los intervalos Bootstrap BCa (95%) proporcionan estimaciones robustas sin distorsión por valores extremos.
